# Lab 5 Working with Data on the Command Line

Lab 5's matrix class is the main assignment. This shorter piece is what we work through in
the lab session, and it is a different skill: finding out what is in a data file, and
getting it into shape, before any Python is involved.

Lecture 11 starts from a real grades file that is a mess — a line of notes where the header
should be, words like `no solution` in columns that should hold numbers, scores written as
`68-sent` — and cleans it with pandas. The same jobs, and almost always the *first* look at
a new file, can be done faster with the Unix tools already on your machine. They also work
when the file is far too big to open.

Lab 3 covered moving around the filesystem: `ls`, `cd`, `mkdir`, `cp`, `mv`. Here we chain
commands together to ask questions of a file and to write a cleaned copy of it.

## The data

Nothing to download. Everything is in the repository you pulled:

| File | What it is |
|---|---|
| `Labs/Lab.5/bank-churn.csv` | 20,000 bank customers: name, country, age, balance, and whether they left. Mixed text and numbers. |
| `Labs/Lab.5/heart.csv` | 303 patients, 14 numeric columns, and a 0/1 diagnosis. |
| `Labs/Lab.5/2012_Workplace_Fatalities_by_State.csv` | Workplace fatalities per US state. Awkward on purpose — see the last exercise. |
| `Lectures/Lecture.11/Data-1401-Grades.csv` | The messy grades file from Lecture 11. |

The bank file is a 20,000-row extract of a Kaggle playground dataset (Binary Classification
with a Bank Churn Dataset, 2024); the heart and fatalities files are included with Lab 5.

## Finding the commands

`man <command>` is the manual for any of these: space scrolls, `q` quits. Some resources
found by searching:

* Paths and wildcards: https://www.warp.dev/terminus/linux-wildcards
* Introduction to the shell: https://github-pages.ucl.ac.uk/RCPSTrainingMaterials/HPCandHTCusingLegion/2_intro_to_shell.html
* Chaining commands: https://www.geeksforgeeks.org/chaining-commands-in-linux/
* Piping: https://www.geeksforgeeks.org/piping-in-unix-or-linux/
* Using `sed`: https://www.geeksforgeeks.org/sed-command-linux-set-2/
* Cheat sheet: https://cheatography.com/davechild/cheat-sheets/linux-command-line/

## How to work

Use a real command prompt: the WSL Ubuntu terminal on Windows, Terminal on macOS. When a
command does what you want, paste it into the cell under the exercise. A cell starting with
`!` runs a shell command from inside Jupyter — that is how Lecture 11 ran `cat` and `wc` —
so your pasted commands will run here too.

Three things to know first:

* `>` sends output into a file, replacing what was there.
* `>>` appends to the end of a file instead.
* `|` (a pipe) feeds one command's output straight into the next.


## Exercise 0: Somewhere to work

Make a directory for this lab inside your fork and copy the four CSV files into it. Work on
the copies, so a mistake never touches the originals and `git status` stays clean.

Hints: `mkdir`, then `cp`. A wildcard saves typing, and `cp` takes several files at once if
the last argument is a directory.

In [2]:
!mkdir -p workdir
!cp *.csv ../../Lectures/Lecture.11/Data-1401-Grades.csv workdir/
!ls workdir/

2012_Workplace_Fatalities_by_State.csv grades_no_notes.csv
bank_churn_clean.csv                   heart_totals.txt
bank_sorted.csv                        heart.csv
bank-churn.csv                         part1.csv
Data-1401-Grades.csv                   part2.csv
german_customers.csv                   part3.csv
grades_clean.csv


## Exercise 1: First look at a file you have never seen

You have `bank-churn.csv` and know nothing about it. Answer these at the prompt:

1. How big is the file, and how many rows does it have?
2. What are the column names, and how many columns are there?
3. What do the first few rows actually look like?

Hints: `du -h` for size, `wc -l` for lines. `head -1` gets the header; piping it into
`tr ',' '\n'` puts each column name on its own line, and piping *that* into `nl` numbers
them — which you will want in the later exercises, where `cut` asks for column numbers.

In [3]:
!du -h workdir/bank-churn.csv
!wc -l workdir/bank-churn.csv

1.4M	workdir/bank-churn.csv
   20001 workdir/bank-churn.csv


In [4]:
!head -1 workdir/bank-churn.csv | tr ',' '\n' | nl

     1	id
     2	CustomerId
     3	Surname
     4	CreditScore
     5	Geography
     6	Gender
     7	Age
     8	Tenure
     9	Balance
    10	NumOfProducts
    11	HasCrCard
    12	IsActiveMember
    13	EstimatedSalary
    14	Exited


In [7]:
!head -5 workdir/bank-churn.csv

id,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,15674932,Okwudilichukwu,668,France,Male,33.0,3,0.0,2,1.0,0.0,181449.97,0
1,15749177,Okwudiliolisa,627,France,Male,33.0,1,0.0,2,1.0,1.0,49503.5,0
2,15694510,Hsueh,678,France,Male,40.0,10,0.0,2,1.0,0.0,184866.69,0
3,15741417,Kao,581,France,Male,34.0,2,148882.54,1,1.0,1.0,84560.88,0


File is ~1.4 MB, 20001 lines (1 header + 20000 rows), 14 columns: `id`, `CustomerId`, `Surname`, `CreditScore`, `Geography`, `Gender`, `Age`, `Tenure`, `Balance`, `NumOfProducts`, `HasCrCard`, `IsActiveMember`, `EstimatedSalary`, `Exited`.

## Exercise 2: Split a big file into three

Files too big to open are routine, and a common first move is to cut one into pieces.

Make 3 new CSV files, each with about a third of `bank-churn.csv`'s rows, and **each
starting with the header line**, so every piece is a readable CSV on its own.

Hints:

* `head -1 ... > part1.csv` starts a file with just the header. Do that three times.
* `wc -l` tells you how many rows there are to divide up. Remember one of them is the header.
* `head -N file | tail -M` selects a range: M lines ending at line N.
* Append each range to the right file with `>>`.
* Check your work: the three files' rows should add back up, `wc -l` on all three at once.

In [8]:
# 20000 data rows split into thirds: 6667 + 6667 + 6666
!head -1 workdir/bank-churn.csv > workdir/part1.csv
!head -1 workdir/bank-churn.csv > workdir/part2.csv
!head -1 workdir/bank-churn.csv > workdir/part3.csv

!head -6668 workdir/bank-churn.csv | tail -6667 >> workdir/part1.csv
!head -13335 workdir/bank-churn.csv | tail -6667 >> workdir/part2.csv
!tail -n +13336 workdir/bank-churn.csv >> workdir/part3.csv

!wc -l workdir/part1.csv workdir/part2.csv workdir/part3.csv

    6668 workdir/part1.csv
    6668 workdir/part2.csv
    6667 workdir/part3.csv
   20003 total


Part 1 and 2 each have 6668 lines (header + 6667 rows), part 3 has 6667 lines (header + 6666 rows). Total data rows: 6667 + 6667 + 6666 = 20000.

## Exercise 3: Select rows by what they contain, and count them

`grep <pattern> <file>` prints the lines that match; `grep -v` prints the ones that don't;
`grep -c` counts instead of printing.

1. Write a CSV holding only the German customers, header line included.
2. What fraction of all customers are German?
3. Of the German customers, what fraction left the bank? The last column, `Exited`, is 1 for
   customers who left. A row for a German customer who left ends in `,1`.

Hints: pipe `grep` into `grep` to require two things at once, and into `wc -l` to count.
Watch out for `Germany` appearing in a surname — check whether that can happen here, and say
how you checked.

In [9]:
!head -1 workdir/bank-churn.csv > workdir/german_customers.csv
!grep "Germany" workdir/bank-churn.csv >> workdir/german_customers.csv
!head -3 workdir/german_customers.csv

id,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
5,15771669,Genovese,588,Germany,Male,36.0,4,131778.58,1,1.0,0.0,136024.31,1
9,15591721,Cattaneo,583,Germany,Male,40.0,4,81274.33,1,1.0,1.0,170843.07,0


In [10]:
!grep -c "Germany" workdir/bank-churn.csv
# total customers = 20000, so fraction = 4155/20000 = ~20.8%

4155


In [11]:
!grep "Germany" workdir/bank-churn.csv | grep -c ",1$"
# 1561 / 4155 = ~37.6% of German customers exited

# check if Germany ever appears as a surname
!grep "Germany" workdir/bank-churn.csv | cut -d',' -f3 | grep -i "germany"

1561


4155 / 20000 = ~20.8% of customers are German. 1561 / 4155 = ~37.6% of German customers exited. The surname check returns nothing so "Germany" never appears in column 3 — the grep is safe.

## Exercise 4: Counting lines is not counting matches

In the grades file, some exam scores are written `68-sent`, the grader's note that the score
had been emailed to the student. Count them.

Careful: `wc -l` counts *lines*, and a line can hold more than one `-sent`. Ask `grep` for
the number of matches rather than the number of matching lines — read `man grep` for `-o`
and for `-c`, and work out which one you need here. Compare the two answers on this file. They agree, which tells you something about the file: no line carries two of them. Describe a row that would make them disagree.

In [12]:
# -c counts matching lines
!grep -c "\-sent" workdir/Data-1401-Grades.csv

# -o prints each match on its own line, so wc -l gives total occurrences
!grep -o "\-sent" workdir/Data-1401-Grades.csv | wc -l

12
      12


Both give 12, so no row has more than one `-sent`. A row like `72-sent,85-sent,0` would make them disagree: `-c` would count it as 1 line, but `-o | wc -l` would count it as 2 matches.

## Exercise 5: Clean a file with `sed`

`sed` replaces text as it passes through. Work on the grades file:

1. Its first line is a row of notes about how each lab was graded, not the header. Write a
   new file with everything **except** that first line.
2. In that new file, make these replacements, and write the result to another CSV:
   * `no solution` becomes empty
   * `did not submit the Lab` becomes empty
   * `did not submit the exam` becomes empty
   * `68-sent` becomes `68`, and likewise for every other score written that way

Hints:

* For part 1, `tail` counts from the start of the file if you ask it the right way:
  `man tail`, look for `-n +K`.
* The form is `sed 's/old/new/g' file`; `g` means every match on the line, not just the first.
* Chain several `sed` commands with pipes.
* The last replacement isn't a fixed string, since the number changes from row to row. You
  want to remove the `-sent` and keep the number, so match only `-sent`.

Compare your part 1 output against `Lectures/Lecture.11/Data-1401-Grades-Fixed.csv`, the
file Lecture 11 hands you: `cmp` or `diff` will tell you whether they're identical. You have
just made it yourself.

This is the same cleaning Lecture 11 does in pandas. Neither way is the right one. The shell
is quicker for a look and a one-off fix; pandas is where you go when the cleaning becomes
part of an analysis you will run again.

In [13]:
# part 1: drop the first line
!tail -n +2 workdir/Data-1401-Grades.csv > workdir/grades_no_notes.csv
!head -3 workdir/grades_no_notes.csv

Lab 1,Lab 2,Lab 3,Lab 4,Exam 1,Lab 5,,Lab 7,Exam 2
,0,0,,0,0,0,0,0
no solution,no solution,-did not submit the Lab,,did not submit the exam,,,,


In [14]:
!cmp workdir/grades_no_notes.csv ../../Lectures/Lecture.11/Data-1401-Grades-Fixed.csv && echo "Files are identical!"

Files are identical!


In [15]:
# part 2: clean the messy values
!sed 's/no solution//g' workdir/grades_no_notes.csv \
  | sed 's/did not submit the Lab//g' \
  | sed 's/did not submit the exam//g' \
  | sed 's/-sent//g' \
  > workdir/grades_clean.csv

!head -5 workdir/grades_clean.csv

Lab 1,Lab 2,Lab 3,Lab 4,Exam 1,Lab 5,,Lab 7,Exam 2
,0,0,,0,0,0,0,0
,,-,,,,,,
100,96,46,,68,,,,
Completed the lab,"ex4- attempted,wrong solution -2


## Exercise 6: Drop the columns that shouldn't be there

`bank-churn.csv` starts with `id`, `CustomerId` and `Surname`. None of them tells you
anything about whether a customer leaves — an id is just bookkeeping, and if a model appears
to learn something from one, it has learned about how the file was built, not about
customers. Dropping them is a real step in preparing data.

Write a new CSV without those three columns.

Hints: `cut -f` selects fields by number and `-d ','` sets the separator, which a CSV needs.
`cut` takes ranges: `-f 4-` means "from the fourth field on".

In [16]:
# columns 1-3 are id, CustomerId, Surname — keep everything from column 4 onward
!cut -d',' -f4- workdir/bank-churn.csv > workdir/bank_churn_clean.csv
!head -3 workdir/bank_churn_clean.csv

CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
668,France,Male,33.0,3,0.0,2,1.0,0.0,181449.97,0
627,France,Male,33.0,1,0.0,2,1.0,1.0,49503.5,0


## Exercise 7: Arithmetic on a column

Use `heart.csv`, which is all numbers.

1. Compute each patient's `trestbps + chol + thalach` (resting blood pressure, cholesterol,
   maximum heart rate) and write the totals to a file.
2. How many patients have `target` equal to 1?

Hints:

* `cut` gets you just the columns you want. Use the numbered header from Exercise 1.
* `tr ',' '+'` turns `145,233,150` into `145+233+150`.
* `bc` is a calculator that reads expressions and prints answers. Pipe into it.
* The header line is text, not numbers, so drop it first, the way you did in Exercise 5.
* `heart.csv` begins with an invisible marker (a byte-order mark) before `age`. It does no
  harm here, but if a command behaves oddly on the first column, that is why.

In [17]:
# trestbps=col4, chol=col5, thalach=col8
# heart.csv uses Windows line endings so tr -d '\r' strips the carriage return
!tail -n +2 workdir/heart.csv \
  | tr -d '\r' \
  | cut -d',' -f4,5,8 \
  | tr ',' '+' \
  | bc \
  > workdir/heart_totals.txt

!head -5 workdir/heart_totals.txt
!wc -l workdir/heart_totals.txt

528
567
506
534
637
     303 workdir/heart_totals.txt


In [18]:
# target is col 14
!tail -n +2 workdir/heart.csv | tr -d '\r' | grep -c ",1$"

165


303 totals written to `heart_totals.txt`. 165 patients have `target = 1`.

## Exercise 8: Sort

1. Sort `bank-churn.csv` by `Balance`, largest first, keeping the header at the top, and
   write the result to a new CSV.
2. Who are the 5 customers with the largest balances, and what countries are they in?

Hints: `sort` with `-n` (numeric), `-r` (reverse), `-t ','` (the separator) and `-k` (which
field). `man sort` is worth reading for `-k`, the fiddly one. Keep the header out of the
sort the same way as before, then put it back with `>` and `>>`.

In [19]:
# Balance is column 9
!head -1 workdir/bank-churn.csv > workdir/bank_sorted.csv
!tail -n +2 workdir/bank-churn.csv \
  | sort -t',' -k9 -nr \
  >> workdir/bank_sorted.csv

!head -3 workdir/bank_sorted.csv

id,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
12268,15757408,Ibezimako,655,Spain,Male,38.0,3,250898.09,3,1.0,1.0,81054.0,0
8949,15587596,Nebeuwa,572,Spain,Male,31.0,5,238387.56,3,1.0,0.0,21646.2,1


In [20]:
# show Surname (col 3), Geography (col 5), Balance (col 9) for top 5
!tail -n +2 workdir/bank_sorted.csv | head -5 | cut -d',' -f3,5,9

Ibezimako,Spain,250898.09
Nebeuwa,Spain,238387.56
Nott,France,222267.63
Thomson,Germany,212696.32
Udinesi,France,212314.03
tail: stdout: Broken pipe


Top 5 by balance: Ibezimako (Spain, 250898.09), Nebeuwa (Spain, 238387.56), Nott (France, 222267.63), Thomson (Germany, 212696.32), Udinesi (France, 212314.03).

## Exercise 9: Where the command line gives up

Open `2012_Workplace_Fatalities_by_State.csv` in a text editor and look at the first few
lines. Then, at the prompt:

1. How many lines does `wc -l` report?
2. How many US states are in the file?
3. Print the first column with `cut -d ',' -f 1`. What do you get?

The two counts in 1 and 2 disagree, and the `cut` output is wrong. Work out why, and write
a sentence or two explaining it. Two things in this file break the assumption every tool you
used today makes, that a line is a row and a comma separates fields.

In [21]:
!wc -l workdir/2012_Workplace_Fatalities_by_State.csv

     105 workdir/2012_Workplace_Fatalities_by_State.csv


In [22]:
!head -6 workdir/2012_Workplace_Fatalities_by_State.csv

State,"Number of Fatalities, 2012","Rate of Fatalities, 2012","State Rank, Fatalities 2012",Number of Injuries/Illnesses 2012,Injuries/Illnesses 2012 Rate,Penalties FY 2013 (Average $),Penalties FY 2013 (Rank),Inspectors,Years to Inspect Each Workplace Once,State or Federal Program
"South Carolina
(33.99882060100049, -81.04536765699964)",63,3.5,25,36200,3,492,49,24,111,State
"West Virginia
(38.665511497000466, -80.71263935099967)",49,6.9,46,19800,4.1,1798,27,7,173,Federal
"Massachusetts


In [23]:
!cut -d',' -f1 workdir/2012_Workplace_Fatalities_by_State.csv | head -10

State
"South Carolina
(33.99882060100049
"West Virginia
(38.665511497000466
"Massachusetts
(42.27687306500047
"Tennessee
(35.680943063000484
"Oklahoma


`wc -l` reports 105 lines but there are only 54 actual data rows (states). `cut` returns partial state names and coordinates instead of clean state names.

Two things break these tools: first, each state cell contains a literal newline inside it (e.g. the coordinates are on a second line inside the same field), so what is one CSV row looks like two lines to `wc`. Second, the column headers and state names contain commas inside double quotes — `cut` does not understand quoting rules and splits on every comma, so it cuts inside the quoted strings and returns the wrong piece. Both are valid CSV that `pandas.read_csv` handles correctly.

## Submitting

Same as every lab: copy this notebook to a new name before working in it, and push the copy
to your fork.

```bash
cp Labs/Lab.5/Lab.5.CommandLine.ipynb Labs/Lab.5/Lab.5.CommandLine.solution.ipynb
```

Each exercise's cell should hold the command or commands you used, and where the exercise
asks a question — a fraction, a count, an explanation — write the answer next to them.

**Do not commit the CSV files you create.** They are working files; your commands are the
submission.
